# ¿Qué es la Computación Vectorizada?

La computación vectorizada consiste en aplicar una operación matemática a un conjunto completo de datos (un "vector" o "matriz") al mismo tiempo, en lugar de iterar elemento por elemento mediante bucles.

Esto aprovecha una capacidad del hardware llamada **SIMD (Single Instruction, Multiple Data)**. En lugar de decirle a la CPU "suma estos dos números" un millón de veces, le dices "suma estas dos listas de un millón de números".

Las operaciones vectoriales y matriciales sin vectorización son lentas debido a que se debe interpretar cada paso del bucle. En consecuencia las operaciones con vectorización son rápidas porque delegas la operación a bibliotecas que ejecutan el trabajo en C o Fortran a bajo nivel.

# Broadcasting: El arte de "estirar" dimensiones

El Broadcasting es una técnica que permite realizar operaciones aritméticas entre arrays de diferentes formas (shapes), siempre que cumplan ciertas reglas de compatibilidad.

Imagina que quieres sumarle 10 puntos de brillo a cada píxel de una imagen. No necesitas crear una imagen del mismo tamaño llena de dieces; simplemente "transmite" (broadcasts) ese valor escalar a través de toda la matriz de la imagen de forma virtual, sin consumir memoria extra.

Regla de oro: Dos dimensiones son compatibles cuando: Son iguales o Una de ellas es 1.

# ¿Cómo ayuda esto en el Procesamiento de Imágenes?

En visión por computadora, una imagen digital no es más que una matriz de números (o un tensor de 3 dimensiones: Alto x Ancho x Canales RGB). 

La vectorización y el broadcasting son fundamentales para: 

* *Ajuste de Brillo y Contraste:* Multiplicar toda la matriz por un factor (contraste) o sumar un valor (brillo) de forma instantánea.
* *Filtros de Color:* Si quieres eliminar el canal rojo de una imagen, simplemente multiplicas el plano R por 0 usando indexación vectorizada.
* *Normalización:* Para redes neuronales, solemos escalar los valores de los píxeles de $[0, 255]$ a $[0, 1]$. Hacer $Imagen / 255.0$ es una operación vectorizada directa.
* *Máscaras y Umbralización (Thresholding):* Crear una imagen binaria haciendo $mascara = imagen > 128$. Esto genera una matriz booleana en milisegundos.

* **Dato curioso:** Si intentas procesar una imagen de alta definición (1080p) con bucles anidados, podrías tardar varios segundos por frame. Con vectorización, puedes procesar decenas de frames por segundo (FPS).

# ¿Qué operaciones podemos realizar? 

En visión por computadora, tratamos a las imágenes como estructuras algebraicas. Al usar computación vectorizada, estas operaciones se ejecutan de forma masiva sobre la malla de píxeles.

Podemos clasificar estas operaciones en tres categorías principales según cómo interactúan con la estructura de la imagen:

1. **Operaciones Punto a Punto (Element-wise)**

Son las más básicas y se benefician directamente del broadcasting. La operación se aplica a cada píxel individualmente sin importar sus vecinos.

* **Ajuste de Brillo:** Sumar un escalar $b$ a la matriz de la imagen $I$.
$$I_n = I+b$$
* **Corrección de Gamma:** Una operación no lineal para ajustar la luminancia.
$$V_{out} = A \cdot V_{in}^{\gamma}$$
* **Mezcla de Imágenes (Blending):** Combinar dos imágenes $I_1$ e $I_2$ usando un factor de transparencia $\alpha$.
$$I_{final} = \alpha I_1 + (1 - \alpha)I_2$$

2. **Operaciones de Vecindad (Convoluciones)**

Aquí es donde ocurre la "magia" de la visión artificial (como en las Redes Neuronales Convolucionales o CNN). No solo importa el píxel, sino también los que están a su alrededor.

Se utiliza un kernel (una matriz pequeña, por ejemplo de $3 \times 3$) que se desliza sobre la imagen. Matemáticamente, es un producto escalar vectorizado en cada posición.

$$G = \sqrt{G_x^2 + G_y^2}$$

3. **Operaciones Morfológicas**

Estas operaciones se basan en la teoría de conjuntos y se aplican generalmente a imágenes binarias (blanco y negro). Utilizan un "elemento estructurante" para modificar la forma de los objetos.

4. **Transformaciones Geométricas (Álgebra Lineal)**

Aquí vemos la vectorización en su máxima expresión mediante matrices de transformación. En lugar de mover cada píxel manualmente, aplicamos una multiplicación matricial a las coordenadas $(x, y)$ de todos los píxeles simultáneamente.

$$\begin{bmatrix} x' \\ y' \end{bmatrix} = M \begin{bmatrix} x \\ y \end{bmatrix}$$

# Operaciones matemáticas

Una imagen no es una *foto*, sino un espacio vectorial discretizado. Para entender la computación vectorizada, debemos ver la imagen como un tensor de orden 3 (o una matriz de orden 2 si es en blanco y negro).

En álgebra lineal pura, una imagen de $M \times N$ píxeles se puede ver de dos formas:

* **Como Matriz:** Una transformación lineal $A \in \mathbb{R}^{M \times N}$.
* **Como Vector (Flattening):** Un solo vector columna $v \in \mathbb{R}^{MN}$.

Donde Cada píxel es un vector en un espacio de color (por ejemplo, el espacio $\mathbb{R}^3$ para RGB). Cuando realizas operaciones entre imágenes, estás haciendo una combinación lineal de vectores: 

$$I_{resultado} = \alpha I_1 + \beta I_2$$

Desde la perspectiva del álgebra lineal pura, operar una imagen (matriz) con un escalar es el ejemplo más directo de espacios vectoriales. Una imagen $I$ de tamaño $m \times n$ pertenece al espacio vectorial de las matrices $\mathbb{R}^{m \times n}$.



## Suma de un Escalar (Desplazamiento o Bias)

Técnicamente, en el álgebra lineal estricta, no puedes sumar un escalar a una matriz ($A + k$ no está definido porque tienen dimensiones distintas). Sin embargo, en computación vectorizada usamos el Broadcasting para simular una suma con una matriz de unos $J$ del mismo tamaño que $A$:

$$A + k \cdot J = [a_{ij} + k]$$
$$A + k \cdot J = \begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} + \begin{bmatrix} k & k \\ k & k \end{bmatrix}$$
$$\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{bmatrix} + k \cdot \begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
\end{bmatrix} = \begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{bmatrix} + \begin{bmatrix}
k * 1 & k * 1 & k * 1 \\
k * 1 & k * 1 & k * 1 \\
\end{bmatrix} = \begin{bmatrix}
1+k & 2+k & 3+k \\
4+k & 5+k & 6+k
\end{bmatrix}$$

*Esto es una traslación en el eje de intensidad. Mueves todos los puntos del espacio de color hacia el blanco (suma) o hacia el negro (resta) de forma uniforme. Es lo que conocemos como Bias en redes neuronales.*

*Cuidado con el Overflow: En computación, si a un píxel de valor 250 le sumas 10, podrías terminar con 4 (si usas uint8 que vuelve a cero) en lugar de 260. Por eso usamos funciones como cv2.add() que "saturan" el valor en 255.*

In [1]:
import numpy as np

A = np.array([[1,2,3],[4,5,6]])
print(f'Matrix A: \n{A}')

k = 10
B = A + k
print(f'Matrix B = A+k: \n{B}')

Matrix A: 
[[1 2 3]
 [4 5 6]]
Matrix B = A+k: 
[[11 12 13]
 [14 15 16]]


## Multiplicación por un Escalar (Escalamiento de Intensidad)

En álgebra lineal, la multiplicación de una matriz $A$ por un escalar $k$ se define como la multiplicación de cada elemento $a_{ij}$ por dicho escalar:

$$k \cdot A = [k \cdot a_{ij}]$$
$$ k \cdot \begin{bmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23}
\end{bmatrix} = \begin{bmatrix}
k \cdot a_{11} & k \cdot a_{12} & k \cdot a_{13} \\
k \cdot a_{21} & k \cdot a_{22} & k \cdot a_{23}
\end{bmatrix}$$

*Esta es una transformación lineal. Si $k > 1$, estás expandiendo el rango de valores (aumentando el contraste); si $0 < k < 1$, estás comprimiendo el rango hacia el negro (oscureciendo y perdiendo contraste).*

In [2]:
A = np.array([[1,2,3],[4,5,6]])
print(f'Matrix A: \n{A}')

k = 10
B = A * k
print(f'Matrix B = A+k: \n{B}')

Matrix A: 
[[1 2 3]
 [4 5 6]]
Matrix B = A+k: 
[[10 20 30]
 [40 50 60]]


## Espacio de Colores como Vector

Si dejamos de ver la imagen como una matriz y nos enfocamos en verla como un conjunto de puntos en un espacio euclidiano tridimensional. Desde el álgebra lineal, cada píxel no es un número, sino un vector columna $v \in \mathbb{R}^3$ dentro del espacio de color (comúnmente RGB).

Imagina un sistema de coordenadas 3D donde los ejes son R (Rojo), G (Verde) y B (Azul). Cualquier color que veas en una pantalla es un vector que nace en el origen $(0,0,0)$ y termina en una coordenada específica.

* El Origen $(0,0,0)$: Representa el vector nulo (Negro).
* El Vértice Opuesto $(255, 255, 255)$: Es el vector del Blanco puro 
* La Diagonal Principal: Todos los vectores donde $R = G = B$ caen sobre esta línea, que representa la escala de grises.

Al tratar los píxeles como vectores, podemos aplicar operaciones de álgebra lineal que tienen efectos visuales inmediatos:

1. **Suma de Vectores (Filtros de Color)**

Si tienes un píxel $v = [100, 50, 0]$ (un tono naranja oscuro) y le sumas un "vector de desplazamiento" $u = [0, 0, 50]$ (azul), el resultado es un nuevo vector:

$$v' = v + u = [100, 50, 50]$$

*Visualmente, esto añade un "tinte" azul a la imagen. En computación vectorizada, sumas el vector $u$ a la matriz de la imagen completa mediante broadcasting, cambiando el balance de blancos de un solo golpe.*

In [3]:
v = np.array([100,50,0])
u = np.array([0,0,50])

v_out = v + u

print(f'vector v_out: {v_out}')

vector v_out: [100  50  50]


2. Producto Escalar (Proyección a Grises)

Como mencionamos antes, la conversión a blanco y negro es una proyección ortogonal. Queremos proyectar el vector 3D $[R, G, B]$ sobre un escalar (una sola dimensión).
Esto se logra mediante el producto punto con un vector de pesos $w$:

$$Luminancia = \vec{v} \cdot \vec{w}$$

*Matemáticamente, estás midiendo qué tanto "pesa" cada componente de color en la percepción humana de la luz.*

A nivel de un solo píxel, tenemos un vector de color $\vec{v}$ con tres componentes: $\vec{v} = [R, G, B]$.Si queremos convertirlo a gris, usamos un vector de pesos $\vec{w} = [0.299, 0.587, 0.114]$. El producto escalar (o producto punto) se define como la suma de los productos de sus componentes:

$$\text{Gris} = \vec{v} \cdot \vec{w} = (R \times 0.299) + (G \times 0.587) + (B \times 0.114)$$

El resultado es un escalar (un solo número). Hemos proyectado un vector 3D a una sola dimensión (intensidad).

In [4]:
v = np.array([100,50,0])
w = np.array([0.299,0.587,0.114])

gris = v*w
print(f'vector gris: {gris}')

vector gris: [29.9  29.35  0.  ]


Aquí es donde entra la computación vectorizada. Una imagen no es un solo vector, sino millones de ellos. Si ponemos todos esos vectores de píxeles uno debajo del otro, formamos una matriz $A$.

$$A = \begin{bmatrix} R_1 & G_1 & B_1 \\ R_2 & G_2 & B_2 \\ \vdots & \vdots & \vdots \\ R_n & G_n & B_n \end{bmatrix}$$

Si queremos calcular el gris de todos los píxeles a la vez, el álgebra lineal nos dice que debemos realizar una multiplicación de matrices. Sin embargo, para que la multiplicación sea válida según las reglas de dimensiones, $A$ con dimensiones $nxm$ debe multiplicarse con $w$ de dimensiones $mxn$, Por eso, trasponemos el vector de pesos a $\vec{w}^T$, convirtiéndolo en un vector columna.

$$A \times \vec{w}^T = \begin{bmatrix} R_1 & G_1 & B_1 \\ R_2 & G_2 & B_2 \end{bmatrix} \times \begin{bmatrix} 0.299 \\ 0.587 \\ 0.114 \end{bmatrix} = \begin{bmatrix} \text{Gris}_1 \\ \text{Gris}_2 \end{bmatrix}$$

In [5]:
A = np.array([[1,2,3],[4,5,6]])
print(f'Matrix A: \n{A}')

w = np.array([0.299,0.587,0.114])

l_out = np.dot(A,w.T)
print(f'salida: {l_out}')


Matrix A: 
[[1 2 3]
 [4 5 6]]
salida: [1.815 4.815]


## Indexación Booleana (o Masking)

La indexación booleana, comúnmente llamada Masking (enmascaramiento), es la técnica de usar una matriz de valores "Verdadero/Falso" para filtrar o modificar elementos de otra matriz.

Desde el álgebra lineal y la computación vectorizada, no es una simple condicional; es una operación de selección masiva que permite aplicar transformaciones solo a los subconjuntos de datos que nos interesan.

Imagina que tienes una hoja de papel con números (tu matriz de imagen $A$) y encima colocas una cartulina negra con agujeros recortados (tu máscara $M$). Solo podrás ver o pintar los números que coinciden con los agujeros. Esto ocurre en dos etapas vectorizadas:

* Se aplica una comparación lógica a toda la matriz. `mask = (A > 128)` Esto genera una nueva matriz de la misma forma que $A$, pero llena de bits (1 para True, 0 para False).
* Se usa esa matriz de bits para indexar. `A[mask] = 255` Aquí, el hardware solo escribe el valor 255 en las direcciones de memoria donde la máscara tiene un 1.

*Matemáticamente, es una operación de Producto de Hadamard encubierta. Si multiplicas tu imagen $A$ por una máscara $M$ (donde True=1 y False=0), los valores que no cumplen la condición se anulan (se vuelven 0).*

Aplicaciones Reales en Visión Artificial:
* Segmentación de Color (Chroma Key): Si quieres quitar un fondo verde, creas una máscara donde los píxeles verdes sean True. Luego, usas esa máscara para reemplazar esos píxeles por los de otra imagen (un paisaje, por ejemplo).
* Umbralización (Thresholding): Es el primer paso para detectar formas. Si quieres separar objetos oscuros de un fondo claro.
* Eliminación de Outliers: En sensores de profundidad (como LiDAR), a veces hay lecturas erróneas de "infinito". Con masking las borras instantáneamente.

*En un bucle tradicional, la CPU tiene que detenerse a evaluar cada píxel: "¿Este es verde? Sí. ¿Este es verde? No...". Esto rompe la "tubería" (pipeline) de procesamiento.Con la indexación booleana, la operación se vuelve predicada. El procesador carga bloques de 16 o 32 píxeles a la vez y aplica la lógica de la máscara en un solo paso eléctrico. Es como pasar de pintar una pared con un pincel minúsculo (bucle for) a usar un rodillo gigante con una plantilla (Masking).*



In [6]:
# tradicional
print("Proceso tradicional")

A = np.array([[0.84, -0.26, 0.92], [0.42, 0.31, -0.73]])
print(f'matriz A: \n{A}')

mask = A < 0
print(f'mask: \n{mask}')

out = A * ~mask
print(f'Salida: \n{out}')

Proceso tradicional
matriz A: 
[[ 0.84 -0.26  0.92]
 [ 0.42  0.31 -0.73]]
mask: 
[[False  True False]
 [False False  True]]
Salida: 
[[ 0.84 -0.    0.92]
 [ 0.42  0.31 -0.  ]]


In [7]:
# Indexación Booleana
print("Indexación Booleana")

A = np.array([[0.84, -0.26, 0.92], [0.42, 0.31, -0.73]])
print(f'matriz A: \n{A}')

A[A < 0] = 0

print(f'matriz A final: \n{A}')

Indexación Booleana
matriz A: 
[[ 0.84 -0.26  0.92]
 [ 0.42  0.31 -0.73]]
matriz A final: 
[[0.84 0.   0.92]
 [0.42 0.31 0.  ]]


## La "navaja suiza" de la toma de decisiones.

la versión vectorizada de la estructura logica if-else de Python, pero diseñada para operar sobre matrices completas a la velocidad del rayo, es la estructura `numpy.where`.

La forma más común de usarlo es: np.where(condición, valor_si_true, valor_si_false).Imagina que tienes dos matrices, $A$ y $B$, y quieres crear una matriz nueva $C$ eligiendo elementos de una u otra según una regla.

Matemáticamente, lo que `np.where hace es una combinación lineal de dos matrices ($X$ y $Y$) utilizando una matriz indicatriz $W$ (la máscara booleana):

$$C = W \odot X + (J - W) \odot Y$$

Donde:
* $W$ es la matriz de unos y ceros (donde $1$ es True).
* $J$ es la matriz de unos (la matriz identidad para el producto de Hadamard).
* $(J - W)$ es la matriz inversa, que tiene $1$ donde $W$ tenía $0$.

*Este diagrama muestra que no hay una "decisión" secuencial, sino una suma de productos que ocurre en paralelo.*

In [8]:
vetor_x = np . array ([1.1 , 1.2 , 1.3 , 1.4 , 1.5])
vetor_y = np . array ([2.1 , 2.2 , 2.3 , 2.4 , 2.5])
cond = np . array ([ True , False , True , True , False ])

resultado = np . where ( cond , vetor_x , vetor_y )
print ( resultado )

[1.1 2.2 1.3 1.4 2.5]


## Ejercicios 

In [9]:
# Inicializar una matriz 5x10 con valor de 7
%time
A = np.ones([5,10]) * 7

print(f'Matriz A: \n{A}')


CPU times: user 6 μs, sys: 0 ns, total: 6 μs
Wall time: 18.8 μs
Matriz A: 
[[7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]
 [7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]
 [7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]
 [7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]
 [7. 7. 7. 7. 7. 7. 7. 7. 7. 7.]]


In [10]:
# Inicializar una matriz 5x10 con valor de 7
%time
A = np.full([5,10],7)

print(f'Matriz A: \n{A}')

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 10 μs
Matriz A: 
[[7 7 7 7 7 7 7 7 7 7]
 [7 7 7 7 7 7 7 7 7 7]
 [7 7 7 7 7 7 7 7 7 7]
 [7 7 7 7 7 7 7 7 7 7]
 [7 7 7 7 7 7 7 7 7 7]]


In [11]:
# vectorize el siguiente código 
%time
np.random.seed(42)
dados = np.random.randn(200,400)
for i in range(200):
    for j in range(400):
        if dados[i][j] > 0:
            dados[i][j] = 0

print(dados)

CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 9.06 μs
[[ 0.         -0.1382643   0.         ...  0.         -0.11453985
   0.        ]
 [-1.59442766 -0.59937502  0.         ...  0.          0.
   0.        ]
 [ 0.         -0.51604473  0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ... -0.92758303 -0.902271
  -0.84440175]
 [ 0.         -0.96097707 -0.34406792 ...  0.         -0.08295827
   0.        ]
 [ 0.         -0.40517858 -1.2957587  ... -0.46895701 -0.00334278
  -0.99348803]]


In [12]:
%time
np.random.seed(42)
dados = np.random.randn(200,400)
dados[dados>0] = 0
print(dados)

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 10.5 μs
[[ 0.         -0.1382643   0.         ...  0.         -0.11453985
   0.        ]
 [-1.59442766 -0.59937502  0.         ...  0.          0.
   0.        ]
 [ 0.         -0.51604473  0.         ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ... -0.92758303 -0.902271
  -0.84440175]
 [ 0.         -0.96097707 -0.34406792 ...  0.         -0.08295827
   0.        ]
 [ 0.         -0.40517858 -1.2957587  ... -0.46895701 -0.00334278
  -0.99348803]]


In [13]:
# dada la matriz imprimir
# los números mayores que 8
# la suma de los números mayores que 8
%time
M = np.array([[13, 24, 1, 8, 15],
             [23, 5, 7, 14, 16],
             [4, 6, 13, 20, 22],
             [10, 12, 19, 21, 3],
             [11, 18, 25, 2, 9]])

M_1 = M[M>8]
print(f'Mayores que 8: {M_1}')
print(f'Mayores que 8: {M_1.size}')

M_2 = np.sum(M_1)
print(f'Suma de los valores > 8: {M_2}')

CPU times: user 6 μs, sys: 0 ns, total: 6 μs
Wall time: 10.7 μs
Mayores que 8: [13 24 15 23 14 16 13 20 22 10 12 19 21 11 18 25  9]
Mayores que 8: 17
Suma de los valores > 8: 285


## Slicing con paso negativo

Si visualizamos la imagen o el array como un bloque de memoria contiguo, lo que estamos haciendo es una inversión de los ejes (flipping). Matemáticamente, es una operación de reordenamiento de los vectores base.

La notación de Python para arrays funciona con tres parámetros:

* Inicio (vacío): Empieza desde el principio (o el final si el paso es negativo).
* Fin (vacío): Ve hasta el final.
* Paso (-1): Indica la dirección y el salto. Un -1 significa "retrocede un elemento a la vez".

Desde el álgebra lineal, invertir un array o una matriz no es magia; es multiplicar la matriz original $A$ por una Matriz de Permutación Especial ($P$) llamada Matriz de Intercambio o Exchange Matrix.Si $A$ es un vector, `a[::-1]` es equivalente a $P \cdot A$, donde $P$ tiene unos en la diagonal antiorogonal:

$$P = \begin{bmatrix} 0 & 0 & 1 \\ 0 & 1 & 0 \\ 1 & 0 & 0 \end{bmatrix}$$

Al multiplicar por esta matriz, las coordenadas del espacio vectorial se invierten: el último elemento pasa a ser el primero, el penúltimo el segundo, y así sucesivamente.

Dependiendo de en qué dimensión apliques el slice, el resultado visual cambia drásticamente:

* En el eje vertical (imagen[::-1, :]): Realizas un Vertical Flip (espejo de arriba hacia abajo). Es una simetría respecto al eje horizontal.
* En el eje horizontal (imagen[:, ::-1]): Realizas un Horizontal Flip (espejo de izquierda a derecha). Es el efecto "espejo" clásico.
* En el eje de canales (imagen[:, :, ::-1]): Esta es vital. Si tienes una imagen en formato BGR (como la lee OpenCV) y la quieres en RGB (como la usa Matplotlib), simplemente inviertes el orden de los canales de color.

*Es una operación de tiempo constante $O(1)$, independientemente de si la imagen pesa 1 MB o 1 GB.*

In [34]:
%time
A = np.array([[1,2,3],
             [4,5,6],
             [7,8,9]])

print(f'Matriz A: \n{A}')

P = np.fliplr(np.eye(3))
print(f'Matiz P: \n{P}')

A_inv = np.dot(P,A)
print(f'Matiz inversa: \n{A_inv}')

CPU times: user 8 μs, sys: 0 ns, total: 8 μs
Wall time: 15.3 μs
Matriz A: 
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Matiz P: 
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
Matiz inversa: 
[[7. 8. 9.]
 [4. 5. 6.]
 [1. 2. 3.]]


In [47]:
%time
A = np.array([[1,2,3],
             [4,5,6],
             [7,8,9]])

print(f'Matriz A: \n{A}')

A_inv = A[::-1]
print(f'Matiz inversa: \n{A_inv}')


CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 8.82 μs
Matriz A: 
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Matiz inversa: 
[[7 8 9]
 [4 5 6]
 [1 2 3]]


## Masked Arrays

Mientras que la indexación booleana que vimos antes `(A[A < 0] = 0)` destruye o modifica los datos originales, un `masked_array` es una estructura que preserva los datos pero le dice a los algoritmos: "Ignora estos valores, no existen para tus cálculos".

Un `masked_array` no es una sola matriz, sino un objeto compuesto por dos matrices gemelas que viajan juntas:

* Data ($A$): La matriz original con todos los valores (incluyendo los "malos" o ruidosos).
* Mask ($M$): Una matriz booleana del mismo tamaño donde:
     * False (o 0) significa: El dato es válido.
     * True (o 1) significa: El dato está enmascarado (inválido).

Matemáticamente, cuando realizas una operación (como una suma o un promedio) sobre un `masked_array`, NumPy aplica una lógica de filtrado previo a nivel de hardware.

Si tienes una imagen y calculas el promedio $\mu$:

* Sin máscara: $\mu = \frac{1}{N} \sum a_{ij}$ (Incluye errores y ruido).
* Con máscara: $\mu = \frac{1}{N_{validos}} \sum (a_{ij} \cdot \neg m_{ij})$

El álgebra lineal aquí es elegante: la máscara actúa como un anulador de dimensiones. Es como si proyectaras tu matriz sobre un subespacio donde las dimensiones "ruidosas" han sido eliminadas sin alterar la estructura de memoria original.

*En el mundo de los tensores (como en Deep Learning), esto es equivalente a usar un Attention Mask, donde le indicas a la red neuronal qué partes de la secuencia o imagen debe ignorar por completo.*


In [72]:
# Obtener la media de cada columna de un vector 2D, ignorando elementos menores o iguale a cero 
%time
A = np.array ([[2.0 , 4.0 , 5.0] , [ -3.0 , 8.0 , -7.0] , [0.0 , 6.0 , 4.0]])
mask = np.ma.masked_array(A, mask=(A<=0))
media = np.ma.mean(mask, axis=0)
print(f'Media: {media}')

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 11 μs
Media: [2.0 6.0 4.5]


In [75]:
%time
A = np.array ([[2.0 , 4.0 , 5.0] , [ -3.0 , 8.0 , -7.0] , [0.0 , 6.0 , 4.0]])
mask = A>0
A[A<0] = 0

media = np.sum(A, axis=0) / np.sum(mask, axis=0)
print(media)


CPU times: user 6 μs, sys: 0 ns, total: 6 μs
Wall time: 11 μs
[2.  6.  4.5]
